# This version checks a new strategy where we are testing statistically signficant features and most promising N derived for forward returns and for creating N lags

# Master Notebook — Execution Filter v2

**Strategy 1 | Long Only | Bull Regime | EMA 14 Stop Loss | Time Exit**

## Changes from V3.6
- adding features to see better result


In [1]:
import pandas as pd
import pandas_ta as ta
import lightgbm as lgb
import numpy as np

# Loading the Data

In [2]:
df = pd.read_csv('btc_1hour_cleaned.csv')

## Setting the correct formats and cleaning Dataframe

In [3]:
df['Open time'] = pd.to_datetime(df['Open time'])
df['Close time'] = pd.to_datetime(df['Close time'])

In [4]:
df = df.set_index('Open time')

# Feature Engineering

## Functions for features: Strategy 1 + regime

In [5]:
# Features of Strategy 1: Trend Momentum
# ROC
def compute_roc(df):
    df['roc_10'] = df['Close'].pct_change(periods=10)
    df['roc_21'] = df['Close'].pct_change(periods=21)
    return df

# MACD Histogram
def compute_macd(df):
    macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
    df['macd_histogram'] = macd['MACDh_12_26_9']
    return df

# ADX - Used as a feature and for regime detection
def compute_adx(df):
    df['adx'] = ta.adx(df['High'], df['Low'], df['Close'], length=14)['ADX_14']
    return df


# Features of Strategy 2: RSI Divergence + Volume
# RSI 14
def compute_rsi_14(df):
    df['rsi_14'] = ta.rsi(df['Close'], length=14)
    return df

# RSI divergence
def compute_rsi_div(df):

    df['price_high'] = df['Close'].rolling(14).max()
    df['price_low']  = df['Close'].rolling(14).min()
    df['rsi_high']   = df['rsi_14'].rolling(14).max()
    df['rsi_low']    = df['rsi_14'].rolling(14).min()

    bearish_div = (df['Close'] == df['price_high']) & (df['rsi_14'] < df['rsi_high'])
    bullish_div = (df['Close'] == df['price_low'])  & (df['rsi_14'] > df['rsi_low'])

    df['rsi_divergence'] = 0
    df.loc[bullish_div, 'rsi_divergence'] = 1
    df.loc[bearish_div, 'rsi_divergence'] = -1

    df.drop(columns=['price_high', 'price_low', 'rsi_high', 'rsi_low'], inplace=True)

    return df

# OBV Change (10 bars)
def compute_obv(df):
    direction = df['Close'].diff(1).apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))
    obv = (df['Volume'] * direction).cumsum()
    df['obv_change'] = obv.diff(10)
    return df

# Volume ROC (10 bars)
def compute_vol_roc(df):
    df['volume_roc'] = df['Volume'].pct_change(periods=10)
    return df


# Features of Strategy 3: Volatility Breakout
# Bolinger bandwidth
def compute_boilinger(df):
    window = 20
    num_std = 2
    df['sma_20'] = df['Close'].rolling(window=window).mean()
    df['sigma'] = df['Close'].rolling(window=window).std()
    df['upper_band'] = df['sma_20'] + (num_std * df['sigma'])
    df['lower_band'] = df['sma_20'] - (num_std * df['sigma'])
    df['bandwidth'] = (df['upper_band'] - df['lower_band']) / df['sma_20']
    return df

# ATR (Average-True-Return)
def  compute_atr(df):
    df['atr_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    return df

# NATR — Normalised ATR
def compute_natr(df):
    df['natr'] = ta.natr(df['High'], df['Low'], df['Close'], length=14)
    return df

# Cyclical time features
def compute_time_features(df):
    df['hour_sin']  = np.sin(2 * np.pi * df.index.hour / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df.index.hour / 24)
    df['dow_sin']   = np.sin(2 * np.pi * df.index.dayofweek / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df.index.dayofweek / 7)
    df['month_sin'] = np.sin(2 * np.pi * df.index.month / 12)
    df['month_cos'] = np.cos(2 * np.pi * df.index.month / 12)
    return df

# Columns used for regime detection only, NOT in feature matrix (x)
# SMA 50 and SMA 200 is purely used for regime detection but not in feature matrix
def compute_regime(df):
    df['sma_50']  = df['Close'].rolling(50).mean()
    df['sma_200'] = df['Close'].rolling(200).mean()
    df['regime']  = (df['sma_50'] > df['sma_200']).astype(int).replace(0, -1)
    return df

# EMA 12 — used as dynamic stop loss in execution layer
def compute_ema12(df):
    df['ema_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    return df

# # EMA 7 — used as dynamic stop loss in execution layer
# def compute_ema7(df):
#     df['ema_7'] = df['Close'].ewm(span=7, adjust=False).mean()
#     return df

## Pipeline

In [6]:
def feature_pipeline(df):
    steps = [
        # model features (go to X)
        compute_roc,
        compute_macd,
        compute_adx,
        compute_rsi_14,
        compute_rsi_div,
        compute_obv,
        compute_vol_roc,
        compute_boilinger,
        compute_atr,
        compute_natr,
        compute_time_features,
        # execution layer only (do not go to X)
        compute_regime,
        compute_ema12,    # new ✅ execution layer only — does NOT go into X
    ]
    for step in steps:
        df = step(df)
    return df

In [7]:
df = feature_pipeline(df)

In [8]:
df.columns

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'Close time',
       'Quote asset volume', 'Number of trades', 'Taker buy base asset volume',
       'Taker buy quote asset volume', 'Ignore', 'roc_10', 'roc_21',
       'macd_histogram', 'adx', 'rsi_14', 'rsi_divergence', 'obv_change',
       'volume_roc', 'sma_20', 'sigma', 'upper_band', 'lower_band',
       'bandwidth', 'atr_14', 'natr', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos', 'month_sin', 'month_cos', 'sma_50', 'sma_200', 'regime',
       'ema_12'],
      dtype='str')

In [9]:
df.shape

(71588, 36)

# Creating labels (y)

In [10]:
# Lookahead window - change this value to test different horizons
N = 12

# Step 1 - Forward return
df['forward_return'] = df['Close'].pct_change(periods=N).shift(-N)

In [11]:
# Step 2 - Binary Label
#old:
# #df['label'] = (df['forward_return'] > 0).astype(int)
#new:
df['label'] = (
    (df['forward_return'] > 0) & (df['regime'] == 1)
).astype(int)
# 1 - BUY, 0 - SELL

# Creating the lags

In [12]:
# ── Lag Creation — Variable Depth per Feature ──────────────────────────────
lag_config = {
    'Volume':         12,
    'roc_10':          5,
    'roc_21':         12,
    'macd_histogram': 12,
    'rsi_14':          5,
    'natr':           12,
    'bandwidth':      12,
    'atr_14':         12,
}

# No-lag features — added directly to X, no temporal history needed
# Time cyclical features are deterministic from timestamp — lags carry no info
no_lag_cols = ['hour_cos', 'month_cos', 'dow_sin']

feature_cols = list(lag_config.keys())

# Build all lag columns at once — avoids DataFrame fragmentation warning
lag_frames = []
all_lag_cols = []

for col, n_lags in lag_config.items():
    for lag in range(1, n_lags + 1):
        col_name = f'{col}_lag{lag}'
        lag_frames.append(df[col].shift(lag).rename(col_name))
        all_lag_cols.append(col_name)

# Join all lags to df in a single operation
df = pd.concat([df] + lag_frames, axis=1)

# Drop NaN rows — no_lag_cols excluded from dropna check
# (they are computed from index so never NaN)
df = df.dropna(subset=feature_cols + all_lag_cols + ['label'])

print(f"Feature columns (base):   {len(feature_cols)}")
print(f"No-lag columns:           {len(no_lag_cols)}")
print(f"Lag columns created:      {len(all_lag_cols)}")
print(f"Total feature columns:    {len(feature_cols) + len(no_lag_cols) + len(all_lag_cols)}")
print(f"Rows after dropna:        {len(df):,}")

Feature columns (base):   8
No-lag columns:           3
Lag columns created:      82
Total feature columns:    93
Rows after dropna:        71,543


In [13]:
# ── Feature Matrix ──────────────────────────────────────────────────────────
# Built from feature_cols (base) + all_lag_cols (variable depth per feature)
# Column list is generated from lag_config — no manual listing needed

X = df[feature_cols + all_lag_cols + no_lag_cols]

print(f"X shape: {X.shape}")

X shape: (71543, 93)


In [14]:
X.columns

Index(['Volume', 'roc_10', 'roc_21', 'macd_histogram', 'rsi_14', 'natr',
       'bandwidth', 'atr_14', 'Volume_lag1', 'Volume_lag2', 'Volume_lag3',
       'Volume_lag4', 'Volume_lag5', 'Volume_lag6', 'Volume_lag7',
       'Volume_lag8', 'Volume_lag9', 'Volume_lag10', 'Volume_lag11',
       'Volume_lag12', 'roc_10_lag1', 'roc_10_lag2', 'roc_10_lag3',
       'roc_10_lag4', 'roc_10_lag5', 'roc_21_lag1', 'roc_21_lag2',
       'roc_21_lag3', 'roc_21_lag4', 'roc_21_lag5', 'roc_21_lag6',
       'roc_21_lag7', 'roc_21_lag8', 'roc_21_lag9', 'roc_21_lag10',
       'roc_21_lag11', 'roc_21_lag12', 'macd_histogram_lag1',
       'macd_histogram_lag2', 'macd_histogram_lag3', 'macd_histogram_lag4',
       'macd_histogram_lag5', 'macd_histogram_lag6', 'macd_histogram_lag7',
       'macd_histogram_lag8', 'macd_histogram_lag9', 'macd_histogram_lag10',
       'macd_histogram_lag11', 'macd_histogram_lag12', 'rsi_14_lag1',
       'rsi_14_lag2', 'rsi_14_lag3', 'rsi_14_lag4', 'rsi_14_lag5', 'natr_lag1',
     

# Exporting the Dataframe with all features and y as a Picklefile

In [15]:
# Only run this when we need to generate the dataframe to be used in our final model
# df.to_pickle('preprocessed.pkl')

In [16]:
# create a function to call this
#pd.read_pickle('../raw_data/preprocessed.pkl').head(3)

# Creating the Feature Matrix and Y

In [17]:
y = df[['label']]

# Training the Model

## Creating the Train / Test split according to user Input

In [18]:
# ── Train / Test Split ─────────────────────────────────────────────────────
CUTOFF_DATE = '2025-01-01'

X_train   = X[X.index < CUTOFF_DATE]
y_train   = y[y.index < CUTOFF_DATE]

X_predict = X[X.index >= CUTOFF_DATE]
y_predict = y[y.index >= CUTOFF_DATE]  # kept for evaluation only

# ── Split diagnostics ──────────────────────────────────────────────────────
total_bars   = len(X)
train_bars   = len(X_train)
test_bars    = len(X_predict)
train_pct    = train_bars / total_bars * 100
test_pct     = test_bars  / total_bars * 100

train_start  = X_train.index.min().strftime('%Y-%m-%d')
train_end    = X_train.index.max().strftime('%Y-%m-%d')
test_start   = X_predict.index.min().strftime('%Y-%m-%d')
test_end     = X_predict.index.max().strftime('%Y-%m-%d')

# Target: train >= 70%, test <= 30%
split_ok = train_pct >= 70.0

print(f"{'='*45}")
print(f"  TRAIN / TEST SPLIT SUMMARY")
print(f"{'='*45}")
print(f"  Train:  {train_bars:>6,} bars  ({train_pct:>5.1f}%)  {train_start} → {train_end}")
print(f"  Test:   {test_bars:>6,} bars  ({test_pct:>5.1f}%)  {test_start} → {test_end}")
print(f"  Total:  {total_bars:>6,} bars")
print(f"{'='*45}")
print(f"  Target: ≥70% train / ≤30% test")
print(f"  Status: {'✅ OK' if split_ok else '❌ WARNING — train below 70%'}")
print(f"{'='*45}")

if not split_ok:
    print(f"\n  ⚠️  Move CUTOFF_DATE earlier to increase training data.")
    print(f"      Current train share: {train_pct:.1f}% (minimum: 70.0%)")

  TRAIN / TEST SPLIT SUMMARY
  Train:  61,201 bars  ( 85.5%)  2018-01-02 → 2024-12-31
  Test:   10,342 bars  ( 14.5%)  2025-01-01 → 2026-03-07
  Total:  71,543 bars
  Target: ≥70% train / ≤30% test
  Status: ✅ OK


## Training the model with LightGBM

In [19]:
# Class imbalance correction
scale_pos_weight = float((y_train['label'] == 0).sum() / (y_train['label'] == 1).sum())

# Settting the parametrs needed for the model
params = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'boosting_type':     'gbdt',
    'learning_rate':     0.05,
    'num_leaves':        31,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.7,
    'bagging_freq':      5,
    'scale_pos_weight':  scale_pos_weight,
    'seed':              42,
    'verbose':           -1,
    'min_child_samples': 20    # was missing
}

In [20]:
# Packaging the training data into LightGBM's required format
train_set_lgb = lgb.Dataset(X_train, label=y_train)

# Training the model
final_model= lgb.train(
    params= params,
    train_set= train_set_lgb,
    num_boost_round= 300,
    callbacks= [lgb.log_evaluation(period=-1)]
)

# Generate predictions on post-cutoff data
pred_proba_final = final_model.predict(X_predict)

## Building the signals DataFrame - this is the input to the execution filter

In [21]:
#new ✅ ema_12
signals_df = df.loc[X_predict.index, ['Close', 'Low', 'sma_50', 'sma_200', 'adx', 'atr_14', 'ema_12']].copy()
signals_df['pred_proba'] = pred_proba_final
signals_df['true_label'] = y_predict.values

# Execution Filter + P&L Tracking

Purpose: Convert raw model probabilities into trade actions bar by bar

Why: Model output alone is not a trade — regime + confidence must confirm

MVP: Long only, bull regime only, confidence threshold filter

Entry metrics (stop distance, position size) are LOCKED at entry bar and never change during the trade — this prevents ATR drift from distorting stop levels and P&L calculations

Expandable: Add short trading in V2 by extending the bear regime block

Exit strategies used:
   1. Stop Loss     — price hits entry_price - trade_stop_dist (hard risk control)
   2. Regime Exit   — market structure no longer supports the trade
   3. Signal Exit   — model confidence drops below (1 - threshold)

In [22]:
# Setting the parameters of the execution strategy
CONFIDENCE_THRESHOLD = 0.58
INITIAL_CAPITAL = 1000.0
ATR_MULTIPLIER = 1.5

# RISK_PCT kept for reference only — position sizing uses 10% of capital
RISK_PCT = 0.01   # risk 1% of capital per trade #RISK_PCT = 0.01
COST_PCT = 0.001  # 0.1% transaction cost per side
#new ✅
POSITION_SIZE = 1  # fraction of capital per trade — change here #new ✅

In [23]:
# Setting the loop that executes the trades

# State variables - maintained by the loop, not stored in Dataframe
position = 'flat'
entry_price = 0.0
trade_pos_size  = 0.0    # locked at entry, never changes mid-trade
trade_stop_dist = 0.0    # locked at entry, never changes mid-trade
capital = INITIAL_CAPITAL
#new ✅: we exit after N bars or after stop loss.
N_HOLD = 12        # exit after N bars
bars_in_trade = 0  # counter — reset at each entry


# Storing the trade logs
trade_log = []

for timestamp, row in signals_df.iterrows():
    close   = row['Close']
    sma_50  = row['sma_50']
    sma_200 = row['sma_200']
    adx     = row['adx']
    atr     = row['atr_14']
    proba   = row['pred_proba']

    # Skip bars where indicators are not yet available (NaN warmup period)
    if pd.isna(atr) or pd.isna(adx) or pd.isna(sma_50) or pd.isna(sma_200):
        continue

    # Regime determination — checked every bar regardless of position
    # Bull requires price above both SMAs AND a strong trend (ADX > 25)
    # ADX < 25 = ranging/choppy market — signals unreliable
    bull = (close > sma_200) and (close > sma_50) and (adx > 25)

    # MANAGEMENT BLOCK — only runs when already in a trade (position == 'long')
    # Checks exit conditions in priority order:
    # Stop Loss → Regime Exit → Signal Exit


    #new ✅
    if position == 'long':

        # EXIT 1: Stop Loss — position closed when Low touches below EMA 7
        # Uses bar Low to detect intrabar breach, not just close price
        #if row['Low'] < row['ema_12']:
        if close < row['ema_12']:
            pnl  = (row['ema_12'] - entry_price) * trade_pos_size
            cost = row['ema_12'] * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp,
                'action':    'stop_loss',
                'close':     row['ema_12'],
                'pnl':       pnl,
                'cost':      cost,#new ✅
                'capital':   capital
            })
            position, entry_price, trade_pos_size, trade_stop_dist, bars_in_trade = 'flat', 0.0, 0.0, 0.0, 0
            continue

        # EXIT 2: Time Exit — close after N bars
        bars_in_trade += 1
        if bars_in_trade >= N_HOLD:
            pnl  = (close - entry_price) * trade_pos_size
            cost = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp,
                'action':    'close_long_time',
                'close':     close,
                'cost':      cost, #new ✅
                'pnl':       pnl,
                'capital':   capital
            })
            position, entry_price, trade_pos_size, trade_stop_dist, bars_in_trade = 'flat', 0.0, 0.0, 0.0, 0
            continue

    else:
        if bull and proba >= CONFIDENCE_THRESHOLD:

            trade_pos_size = (capital * POSITION_SIZE) / close
            # XX trade_pos_size  = (capital * 1.0) / close #new ✅

            cost = close * trade_pos_size * COST_PCT
            capital -= cost
            position = 'long'
            entry_price = close
            bars_in_trade = 0 #new ✅

            trade_log.append({
                'timestamp':  timestamp,
                'action':     'enter_long',
                'close':      close,
                'pnl':        0,
                'cost':       cost, #new ✅
                'capital':    capital
            })

## Performance Summary

In [24]:
import numpy as np

# ================================================================
# FULL PERFORMANCE EVALUATION
# ================================================================

# --- Setup ---
trade_df       = pd.DataFrame(trade_log)
entries        = trade_df[trade_df['action'] == 'enter_long']
exits          = trade_df[trade_df['action'] != 'enter_long']
winning_trades = exits[exits['pnl'] > 0]
losing_trades  = exits[exits['pnl'] < 0]
total_closed   = len(exits)
win_rate       = len(winning_trades) / total_closed * 100 if total_closed > 0 else 0
loss_rate      = len(losing_trades)  / total_closed * 100 if total_closed > 0 else 0

# --- Risk Metrics ---
trade_df_sorted  = trade_df.sort_values('timestamp')
trade_df_sorted['capital_return'] = trade_df_sorted['capital'].pct_change()
mean_return      = trade_df_sorted['capital_return'].mean()
std_return       = trade_df_sorted['capital_return'].std()

# --- Backtest Window ---
backtest_start   = trade_df_sorted['timestamp'].iloc[0].strftime('%d %b %Y')
backtest_end     = trade_df_sorted['timestamp'].iloc[-1].strftime('%d %b %Y')
days_in_backtest = (trade_df_sorted['timestamp'].iloc[-1] - trade_df_sorted['timestamp'].iloc[0]).days
annualised_return = ((capital / INITIAL_CAPITAL) ** (365 / days_in_backtest) - 1) * 100

# ── Corrected Sharpe Ratio ─────────────────────────────────────────────────
# Risk-free rate: 4.25% annualised — approximates average US Federal Funds
# effective rate over the Jan 2024 – Mar 2026 evaluation window
# Annualisation: based on actual average trade frequency (not assumed hourly)
# This replaces the previous sqrt(8760) approach which was overstated

RISK_FREE_RATE  = 0.0425   # 4.25% annualised

# Compute average holding period in hours from actual trade timestamps
entry_times = trade_df[trade_df['action'] == 'enter_long']['timestamp'].values
exit_times  = trade_df[trade_df['action'] != 'enter_long']['timestamp'].values

if len(entry_times) == len(exit_times) and len(entry_times) > 0:
    holding_hours   = np.mean([
        (pd.Timestamp(ex) - pd.Timestamp(en)).total_seconds() / 3600
        for en, ex in zip(entry_times, exit_times)
    ])
    trades_per_year = 8760 / holding_hours
else:
    # Fallback if entries/exits are unbalanced — use trade count over period
    trades_per_year = total_closed / (days_in_backtest / 365)
    holding_hours   = 8760 / trades_per_year

# Risk-free rate per trade observation
rf_per_trade  = RISK_FREE_RATE / trades_per_year

# Sharpe — excess return per unit of risk, annualised at trade frequency
sharpe        = ((mean_return - rf_per_trade) / std_return) * np.sqrt(trades_per_year)

# ── End Sharpe ─────────────────────────────────────────────────────────────

trade_df_sorted['cummax']   = trade_df_sorted['capital'].cummax()
trade_df_sorted['drawdown'] = (trade_df_sorted['capital'] - trade_df_sorted['cummax']) / trade_df_sorted['cummax']
max_drawdown  = trade_df_sorted['drawdown'].min() * 100
gross_profit  = winning_trades['pnl'].sum()
gross_loss    = losing_trades['pnl'].abs().sum()
profit_factor = gross_profit / gross_loss if gross_loss > 0 else 0

# --- Buy and Hold Benchmark ---
buy_date    = CUTOFF_DATE
buy_price   = df.loc[buy_date:, 'Close'].iloc[0]
final_price = df['Close'].iloc[-1]
btc_units   = INITIAL_CAPITAL / buy_price
bnh_value   = btc_units * final_price
bnh_return  = ((bnh_value - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100

# --- Transaction Costs ---
total_transaction_costs = trade_df['cost'].sum()

# --- Position Sizing Analysis ---
entry_trades    = trade_df[trade_df['action'] == 'enter_long'].copy()
avg_capital     = (INITIAL_CAPITAL + capital) / 2
avg_pos_usd     = avg_capital * POSITION_SIZE
avg_pos_btc     = avg_pos_usd / entry_trades['close'].mean()
avg_risk_dollar = avg_pos_usd

# --- Print All Results ---
print(f'╔══════════════════════════════════════╗')
print(f'║         BACKTEST RESULTS             ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ CAPITAL                              ║')
print(f'║  Period: {backtest_start} → {backtest_end}   ║')
print(f'║  Initial:         ${INITIAL_CAPITAL:>10,.2f}        ║')
print(f'║  Final:           ${capital:>10,.2f}        ║')
print(f'║  Total Return:    {((capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100):>9.2f}%         ║')
print(f'║  Annualised:      {annualised_return:>9.2f}%         ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ TRADE STATISTICS                     ║')
print(f'║  Total Trades:    {len(entries):>10}         ║')
print(f'║  Winning Trades:  {len(winning_trades):>10}         ║')
print(f'║  Losing Trades:   {len(losing_trades):>10}         ║')
print(f'║  Win Rate:        {win_rate:>9.1f}%         ║')
print(f'║  Loss Rate:       {loss_rate:>9.1f}%         ║')
print(f'║  Avg Win PnL:     ${winning_trades["pnl"].mean():>10.2f}        ║')
print(f'║  Avg Loss PnL:    ${losing_trades["pnl"].mean():>10.2f}        ║')
print(f'║  Transaction Costs: ${total_transaction_costs:>8.2f}        ║')
print(f'║  Avg Risk per Trade:${avg_risk_dollar:>9.2f}       ║')
print(f'║  Avg Position (BTC):{avg_pos_btc:>9.6f}        ║')
print(f'║  Avg Position ($):  ${avg_pos_usd:>9.2f}       ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ RISK METRICS                         ║')
print(f'║  Sharpe Ratio:    {sharpe:>10.2f}         ║')
print(f'║  Avg Hold (hrs):  {holding_hours:>10.1f}         ║')
print(f'║  RF Rate:              4.25%         ║')
print(f'║  Max Drawdown:    {max_drawdown:>9.2f}%         ║')
print(f'║  Profit Factor:   {profit_factor:>10.2f}         ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ BUY & HOLD BENCHMARK                 ║')
print(f'║  Buy Price:       ${buy_price:>10,.2f}        ║')
print(f'║  Final Price:     ${final_price:>10,.2f}        ║')
print(f'║  Final Value:     ${bnh_value:>10,.2f}        ║')
print(f'║  B&H Return:      {bnh_return:>9.2f}%         ║')
print(f'║  Strategy Return: {((capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100):>9.2f}%         ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ ACTION BREAKDOWN                     ║')
for action, count in trade_df['action'].value_counts().items():
    print(f'║  {action:<20} {count:>6}         ║')
print(f'╚══════════════════════════════════════╝')

╔══════════════════════════════════════╗
║         BACKTEST RESULTS             ║
╠══════════════════════════════════════╣
║ CAPITAL                              ║
║  Period: 02 Jan 2025 → 05 Mar 2026   ║
║  Initial:         $  1,000.00        ║
║  Final:           $  1,162.01        ║
║  Total Return:        16.20%         ║
║  Annualised:          13.70%         ║
╠══════════════════════════════════════╣
║ TRADE STATISTICS                     ║
║  Total Trades:           163         ║
║  Winning Trades:         116         ║
║  Losing Trades:           47         ║
║  Win Rate:             71.2%         ║
║  Loss Rate:            28.8%         ║
║  Avg Win PnL:     $      6.19        ║
║  Avg Loss PnL:    $     -4.28        ║
║  Transaction Costs: $  354.76        ║
║  Avg Risk per Trade:$  1081.01       ║
║  Avg Position (BTC): 0.010464        ║
║  Avg Position ($):  $  1081.01       ║
╠══════════════════════════════════════╣
║ RISK METRICS                         ║
║  Sharpe Ratio: